# GE-YOLOv8 Benchmark on RSNA 2024 Axial T2 Dataset

This notebook trains the custom **GE-YOLOv8** model (from the paper "Deep learning-based automatic detection and grading of disk herniation") on the **RSNA 2024 Lumbar Spine Degenerative Classification** dataset.

**Key Features:**
- **Custom Architecture:** Uses CSP (Gradient Search) and ECAAttention (Efficient Channel Attention) modules
- **Data Pipeline:** Identical to YOLOv11 benchmark for fair comparison
- **3 Class Detection:** Spinal Canal Stenosis (Center), Left (Neural Foraminal & Subarticular), Right (Neural Foraminal & Subarticular)

**References:**
- [Yolov8-GS-ECA GitHub Repository](https://github.com/hxxbb/Yolov8-GS-ECA)
- [RSNA 2024 Lumbar Spine Competition](https://www.kaggle.com/competitions/rsna-2024-lumbar-spine-degenerative-classification)

## 1. Environment Setup

The GE-YOLOv8 model uses custom modules (`CSP` for Gradient Search, `ECAAttention` for Attention) that are NOT in the standard pip package. We need to clone the official repository and install it in editable mode.

In [1]:
#!rm -rf /kaggle/working/Yolov8-GS-ECA
# Clone the GE-YOLOv8 repository
!git clone https://github.com/hxxbb/Yolov8-GS-ECA.git

# 1. Cài đặt bản chuẩn để lấy khung chương trình (class YOLO, engine train...)
!pip install ultralytics==8.0.200

# 2. Tìm đường dẫn nơi thư viện vừa được cài
import ultralytics
import os
import shutil

installed_path = os.path.dirname(ultralytics.__file__)
print(f"Thư viện chuẩn đang nằm tại: {installed_path}")

Cloning into 'Yolov8-GS-ECA'...
remote: Enumerating objects: 126, done.
remote: Counting objects: 100% (126/126), done.
remote: Compressing objects: 100% (121/121), done.
remote: Total 126 (delta 24), reused 6 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (126/126), 240.64 KiB | 1.84 MiB/s, done.
Resolving deltas: 100% (24/24), done.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 644.5/644.5 kB 18.1 MB/s eta 0:00:0000:01
Thư viện chuẩn đang nằm tại: /usr/local/lib/python3.12/dist-packages/ultralytics


In [2]:
import os
import shutil
import ultralytics

# 1. Xác định vị trí thư viện vừa cài lại
installed_path = os.path.dirname(ultralytics.__file__)
print(f"Thư viện cài tại: {installed_path}")

# 2. Đường dẫn tới code custom của bạn (SỬA LẠI đường dẫn này nếu cần)
# Ví dụ: /kaggle/working/Yolov8-GS-ECA-b2e...
custom_repo_path = '/kaggle/working/Yolov8-GS-ECA'

# 3. Chỉ copy đè thư mục 'nn'
src_nn = os.path.join(custom_repo_path, 'nn')
dst_nn = os.path.join(installed_path, 'nn')

if os.path.exists(src_nn):
    print(f"Đang cập nhật code custom từ {src_nn} vào thư viện...")
    if os.path.exists(dst_nn):
        shutil.rmtree(dst_nn) # Xóa nn cũ
    shutil.copytree(src_nn, dst_nn) # Copy nn mới vào
    print("✅ Đã cập nhật thành công thư mục 'nn'.")
else:
    print("❌ Lỗi: Không tìm thấy thư mục 'nn' trong đường dẫn code của bạn.")

Thư viện cài tại: /usr/local/lib/python3.12/dist-packages/ultralytics
Đang cập nhật code custom từ /kaggle/working/Yolov8-GS-ECA/nn vào thư viện...
✅ Đã cập nhật thành công thư mục 'nn'.


In [3]:
import os
import ultralytics

# 1. Xác định đường dẫn file block.py trong thư viện đã cài
ultralytics_path = os.path.dirname(ultralytics.__file__)
block_path = os.path.join(ultralytics_path, 'nn', 'modules', 'block.py')

print(f"Đang sửa file: {block_path}")

# 2. Đọc nội dung file
with open(block_path, 'r') as f:
    lines = f.readlines()

# 3. Tìm và comment dòng import gây lỗi
new_lines = []
modified = False
for line in lines:
    # Tìm dòng chứa lệnh import RFCAConv
    if 'from .RFCAConv import RFCAConv' in line:
        # Thêm dấu # để comment dòng này lại
        new_lines.append(f"# {line}")
        modified = True
        print("✅ Đã tìm thấy và comment dòng lệnh import lỗi: RFCAConv")
    else:
        new_lines.append(line)

# 4. Ghi lại file nếu có thay đổi
if modified:
    with open(block_path, 'w') as f:
        f.writelines(new_lines)
    print("🎉 Sửa lỗi thành công! Bạn có thể import YOLO ngay bây giờ.")
else:
    print("⚠️ Không tìm thấy dòng import RFCAConv. Có thể file đã được sửa hoặc nội dung khác dự kiến.")

Đang sửa file: /usr/local/lib/python3.12/dist-packages/ultralytics/nn/modules/block.py
✅ Đã tìm thấy và comment dòng lệnh import lỗi: RFCAConv
🎉 Sửa lỗi thành công! Bạn có thể import YOLO ngay bây giờ.


In [4]:
# Disable WandB to prevent login prompts that block training
import os
os.environ['WANDB_DISABLED'] = 'true'
os.environ['WANDB_MODE'] = 'disabled'

from ultralytics import YOLO

In [5]:
# Install additional dependencies
!pip install pydicom pandas numpy opencv-python-headless pyyaml tqdm

# Gỡ bỏ ray để tránh Ultralytics tự động gọi nó
!pip uninstall -y ray

Found existing installation: ray 2.53.0
Uninstalling ray-2.53.0:
  Successfully uninstalled ray-2.53.0


In [6]:
# Verification: Ensure ultralytics imports from the local directory
import ultralytics
print(f"Ultralytics version: {ultralytics.__version__}")
print(f"Ultralytics location: {ultralytics.__file__}")

# Verify custom modules are available
# from ultralytics.nn.modules import ECAAttention, CSP
# print("\n✅ Custom modules (ECAAttention, CSP) imported successfully!")

Ultralytics version: 8.0.200
Ultralytics location: /usr/local/lib/python3.12/dist-packages/ultralytics/__init__.py


In [7]:
# Import all required libraries
import os
import glob
import shutil
import random
from pathlib import Path

import numpy as np
import pandas as pd
import cv2
import pydicom
from tqdm import tqdm
import yaml

from ultralytics import YOLO

# Set seeds for reproducibility
random.seed(42)
np.random.seed(42)

## 2. Data Pipeline (RSNA 2024 Axial T2)

This data pipeline is **IDENTICAL** to the YOLOv11 experiment for fair comparison.

**Key Processing Steps:**
- **Filter:** Only `series_description == 'Axial T2'`
- **Resize:** 384 x 384
- **Class Mapping (3 Classes):**
  - Class 0: Spinal Canal Stenosis (Center)
  - Class 1: Left Neural Foraminal & Left Subarticular (Left)
  - Class 2: Right Neural Foraminal & Right Subarticular (Right)

### 2.1 Configuration

In [8]:
# Configuration
class Config:
    # Paths - Kaggle specific
    COMPETITION_DATA = '/kaggle/input/rsna-2024-lumbar-spine-degenerative-classification'
    OUTPUT_DIR = '/kaggle/working/datasets/axial_t2'
    
    # Data processing
    TARGET_SIZE = (384, 384)  # Same as YOLOv11 benchmark
    BBOX_SIZE = 32  # Fixed bounding box size in pixels
    SERIES_FILTER = 'Axial T2'
    
    # Train/Val split
    VAL_RATIO = 0.2
    
    # Class mapping: 3 classes
    # Class 0: Spinal Canal Stenosis (Center)
    # Class 1: Left Neural Foraminal & Left Subarticular (Left)
    # Class 2: Right Neural Foraminal & Right Subarticular (Right)
    CLASS_MAPPING = {
    # Class 0: Center
    'Spinal Canal Stenosis': 0,
    
    # Class 1: Left
    'Left Neural Foraminal Narrowing': 1,
    'Left Subarticular Stenosis': 1,
    
    # Class 2: Right
    'Right Neural Foraminal Narrowing': 2,
    'Right Subarticular Stenosis': 2
    }
    
    CLASS_NAMES = ['Center', 'Left', 'Right']
    NUM_CLASSES = 3

cfg = Config()

### 2.2 Load Metadata

In [9]:
# Load CSV files
train_df = pd.read_csv(f'{cfg.COMPETITION_DATA}/train.csv')
train_label_coords = pd.read_csv(f'{cfg.COMPETITION_DATA}/train_label_coordinates.csv')
train_series_desc = pd.read_csv(f'{cfg.COMPETITION_DATA}/train_series_descriptions.csv')

print(f"Train samples: {len(train_df)}")
print(f"Label coordinates: {len(train_label_coords)}")
print(f"Series descriptions: {len(train_series_desc)}")

Train samples: 1975
Label coordinates: 48692
Series descriptions: 6294


In [10]:
# Filter for Axial T2 series only
axial_t2_series = train_series_desc[train_series_desc['series_description'] == cfg.SERIES_FILTER]
print(f"\nAxial T2 series count: {len(axial_t2_series)}")

# Merge to get label coordinates for Axial T2 series
axial_t2_labels = train_label_coords.merge(
    axial_t2_series[['study_id', 'series_id']], 
    on=['study_id', 'series_id'], 
    how='inner'
)
print(f"Axial T2 labels: {len(axial_t2_labels)}")


Axial T2 series count: 2340
Axial T2 labels: 19220


### 2.3 DICOM Processing Functions

In [11]:
def load_dicom(dicom_path):
    """
    Load DICOM file and normalize to 8-bit.
    
    Args:
        dicom_path: Path to DICOM file
        
    Returns:
        Normalized 8-bit numpy array
    """
    dcm = pydicom.dcmread(dicom_path)
    img = dcm.pixel_array.astype(np.float32)
    
    # Normalize to 0-255
    img = (img - img.min()) / (img.max() - img.min() + 1e-8) * 255
    img = img.astype(np.uint8)
    
    return img


def resize_image(img, target_size):
    """
    Resize image to target size.
    
    Args:
        img: Input image
        target_size: Tuple (width, height)
        
    Returns:
        Resized image and scale factors
    """
    h, w = img.shape[:2]
    resized = cv2.resize(img, target_size, interpolation=cv2.INTER_LINEAR)
    scale_x = target_size[0] / w
    scale_y = target_size[1] / h
    return resized, scale_x, scale_y


def point_to_bbox(x, y, img_width, img_height, bbox_size):
    """
    Convert point coordinates to YOLO format bounding box.
    
    Args:
        x, y: Point coordinates
        img_width, img_height: Image dimensions
        bbox_size: Fixed bounding box size in pixels
        
    Returns:
        YOLO format bbox (x_center, y_center, width, height) - normalized
    """
    # Center coordinates (normalized)
    x_center = x / img_width
    y_center = y / img_height
    
    # Box dimensions (normalized)
    box_w = bbox_size / img_width
    box_h = bbox_size / img_height
    
    # Clamp to valid range
    x_center = max(0, min(1, x_center))
    y_center = max(0, min(1, y_center))
    box_w = min(box_w, min(x_center, 1 - x_center) * 2)
    box_h = min(box_h, min(y_center, 1 - y_center) * 2)
    
    return x_center, y_center, box_w, box_h


def get_condition_class(condition):
    """
    Map condition name to class ID.
    
    Class mapping:
    - 0: Spinal Canal Stenosis (Center)
    - 1: Left Neural Foraminal & Left Subarticular (Left)
    - 2: Right Neural Foraminal & Right Subarticular (Right)
    """
    # condition_lower = condition.lower().replace(' ', '_')
    return cfg.CLASS_MAPPING.get(condition, None)

### 2.4 Process Dataset

In [12]:
def process_dataset(axial_t2_labels, cfg):
    """
    Process the RSNA dataset and create YOLO format dataset.
    
    Label propagation: Labels are shared to slice n-1 and n+1.
    """
    # Create output directories
    for split in ['train', 'val']:
        os.makedirs(f'{cfg.OUTPUT_DIR}/{split}/images', exist_ok=True)
        os.makedirs(f'{cfg.OUTPUT_DIR}/{split}/labels', exist_ok=True)
    
    # Group labels by study and series
    grouped = axial_t2_labels.groupby(['study_id', 'series_id'])
    
    # Get unique study IDs for train/val split
    study_ids = axial_t2_labels['study_id'].unique().tolist()
    random.shuffle(study_ids)
    
    val_size = int(len(study_ids) * cfg.VAL_RATIO)
    val_study_ids = set(study_ids[:val_size])
    train_study_ids = set(study_ids[val_size:])
    
    print(f"Train studies: {len(train_study_ids)}, Val studies: {len(val_study_ids)}")
    
    stats = {'train': {'images': 0, 'labels': 0}, 'val': {'images': 0, 'labels': 0}}
    
    for (study_id, series_id), group in tqdm(grouped, desc="Processing series"):
        # Determine split
        split = 'val' if study_id in val_study_ids else 'train'
        
        # Get series path
        series_path = f"{cfg.COMPETITION_DATA}/train_images/{study_id}/{series_id}"
        
        if not os.path.exists(series_path):
            continue
        
        # Get all DICOM files in series
        dicom_files = sorted(glob.glob(f"{series_path}/*.dcm"))
        if not dicom_files:
            continue
        
        # Create instance number to file mapping
        instance_to_file = {}
        for dcm_path in dicom_files:
            try:
                dcm = pydicom.dcmread(dcm_path, stop_before_pixels=True)
                instance_num = int(dcm.InstanceNumber)
                instance_to_file[instance_num] = dcm_path
            except Exception:
                continue
        
        # Group labels by instance number
        instance_labels = {}
        for _, row in group.iterrows():
            instance_num = int(row['instance_number'])
            
            # Propagate labels to n-1, n, n+1
            for offset in [-1, 0, 1]:
                target_instance = instance_num + offset
                if target_instance not in instance_labels:
                    instance_labels[target_instance] = []
                
                class_id = get_condition_class(row['condition'])
                if class_id is not None:
                    instance_labels[target_instance].append({
                        'class_id': class_id,
                        'x': row['x'],
                        'y': row['y']
                    })
        
        # Process each instance with labels
        for instance_num, labels in instance_labels.items():
            if instance_num not in instance_to_file:
                continue
            
            dcm_path = instance_to_file[instance_num]
            
            try:
                # Load and process image
                img = load_dicom(dcm_path)
                orig_h, orig_w = img.shape[:2]
                
                # Resize
                img_resized, scale_x, scale_y = resize_image(img, cfg.TARGET_SIZE)
                
                # Convert grayscale to RGB
                if len(img_resized.shape) == 2:
                    img_resized = cv2.cvtColor(img_resized, cv2.COLOR_GRAY2RGB)
                
                # Generate unique filename
                filename = f"{study_id}_{series_id}_{instance_num}"
                
                # Save image
                img_path = f"{cfg.OUTPUT_DIR}/{split}/images/{filename}.jpg"
                cv2.imwrite(img_path, img_resized)
                stats[split]['images'] += 1
                
                # Create YOLO format labels
                label_lines = []
                for label in labels:
                    # Scale coordinates
                    x_scaled = label['x'] * scale_x
                    y_scaled = label['y'] * scale_y
                    
                    # Convert to YOLO bbox format
                    x_center, y_center, box_w, box_h = point_to_bbox(
                        x_scaled, y_scaled,
                        cfg.TARGET_SIZE[0], cfg.TARGET_SIZE[1],
                        cfg.BBOX_SIZE
                    )
                    
                    label_lines.append(f"{label['class_id']} {x_center:.6f} {y_center:.6f} {box_w:.6f} {box_h:.6f}")
                    stats[split]['labels'] += 1
                
                # Save labels
                label_path = f"{cfg.OUTPUT_DIR}/{split}/labels/{filename}.txt"
                with open(label_path, 'w') as f:
                    f.write('\n'.join(label_lines))
                    
            except Exception as e:
                print(f"Error processing {dcm_path}: {e}")
                continue
    
    return stats

In [13]:
# Process the dataset
print("Processing RSNA 2024 Axial T2 dataset...")
stats = process_dataset(axial_t2_labels, cfg)

print("\n" + "="*50)
print("Dataset Processing Complete!")
print("="*50)
print(f"\nTrain set: {stats['train']['images']} images, {stats['train']['labels']} labels")
print(f"Val set: {stats['val']['images']} images, {stats['val']['labels']} labels")

Processing RSNA 2024 Axial T2 dataset...
Train studies: 1580, Val studies: 394


Processing series: 100%|██████████| 2339/2339 [20:15<00:00,  1.92it/s]


Dataset Processing Complete!

Train set: 27962 images, 46016 labels
Val set: 6873 images, 11502 labels


## 3. Model Configuration

Create the `data.yaml` file and load the custom GE-YOLOv8 architecture.

### 3.1 Create data.yaml

In [14]:
# Create data.yaml for YOLO training
data_yaml = {
    'path': cfg.OUTPUT_DIR,
    'train': 'train/images',
    'val': 'val/images',
    'nc': cfg.NUM_CLASSES,
    'names': cfg.CLASS_NAMES
}

data_yaml_path = f'{cfg.OUTPUT_DIR}/data.yaml'
with open(data_yaml_path, 'w') as f:
    yaml.dump(data_yaml, f, default_flow_style=False)

print(f"Created data.yaml at: {data_yaml_path}")
print("\nContents:")
with open(data_yaml_path, 'r') as f:
    print(f.read())

Created data.yaml at: /kaggle/working/datasets/axial_t2/data.yaml

Contents:
names:
- Center
- Left
- Right
nc: 3
path: /kaggle/working/datasets/axial_t2
train: train/images
val: val/images



### 3.2 Load Custom GE-YOLOv8 Architecture

The custom model uses:
- **CSP** (Cross Stage Partial) blocks for Gradient Search
- **ECAAttention** (Efficient Channel Attention) for attention mechanism

In [15]:
# Path to custom model YAML
MODEL_YAML = '/kaggle/working/Yolov8-GS-ECA/models/v8/yolov8-CSP.yaml'

# Verify the model config exists
if os.path.exists(MODEL_YAML):
    print(f"✅ Found custom model config: {MODEL_YAML}")
    with open(MODEL_YAML, 'r') as f:
        print("\nModel Architecture:")
        print(f.read())
else:
    print(f"❌ Model config not found: {MODEL_YAML}")
    print("Make sure to clone the Yolov8-GS-ECA repository first!")

✅ Found custom model config: /kaggle/working/Yolov8-GS-ECA/models/v8/yolov8-CSP.yaml

Model Architecture:
# Ultralytics YOLO 🚀, AGPL-3.0 license
# YOLOv8 object detection model with P3-P5 outputs. For Usage examples see https://docs.ultralytics.com/tasks/detect

# Parameters
nc: 10  # number of classes
scales:
#depth: 1.00  #用来控制模型的深度，仅在repeat≠1的时候启用
#width: 1.25  #用来控制模型的宽度，主要作用于args中的out_channel   比如outchannel=64,实际的out_chanel=64x0.25=16通道#scales:
# model compound scaling constants, i.e. 'model=yolov8n.yaml' will call yolov8.yaml with scale 'n'
  # [depth, width, max_channels]
  n: [0.33, 0.25, 1024]  # YOLOv8n summary: 225 layers,  3157200 parameters,  3157184 gradients,   8.9 GFLOPs
  s: [0.33, 0.50, 1024]  # YOLOv8s summary: 225 layers, 11166560 parameters, 11166544 gradients,  28.8 GFLOPs
  m: [0.67, 0.75, 768]   # YOLOv8m summary: 295 layers, 25902640 parameters, 25902624 gradients,  79.3 GFLOPs
  l: [1.00, 1.00, 512]   # YOLOv8l summary: 365 layers, 43691520 parameters, 4369150

In [16]:
# Modify the model YAML to use nc: 3 (our number of classes)
# We create a temporary modified config

with open(MODEL_YAML, 'r') as f:
    model_config = yaml.safe_load(f)

# Update number of classes
model_config['nc'] = cfg.NUM_CLASSES

# Save modified config
modified_model_yaml = f'{cfg.OUTPUT_DIR}/yolov8-CSP-3classn.yaml'
with open(modified_model_yaml, 'w') as f:
    yaml.dump(model_config, f, default_flow_style=False)

print(f"Created modified model config: {modified_model_yaml}")
print(f"Number of classes: {model_config['nc']}")

Created modified model config: /kaggle/working/datasets/axial_t2/yolov8-CSP-3classn.yaml
Number of classes: 3


In [17]:
import torch
import torch.nn as nn
import contextlib
import ast

# Import các thành phần cơ bản từ thư viện chuẩn
from ultralytics.nn import tasks
from ultralytics.nn.modules import (AIFI, C1, C2, C3, C3TR, SPP, SPPF, Bottleneck, BottleneckCSP, C2f, C3Ghost, C3x,
                                    Classify, Concat, Conv, Conv2, ConvTranspose, Detect, DWConv, DWConvTranspose2d,
                                    Focus, GhostBottleneck, GhostConv, HGBlock, HGStem, Pose, RepC3, RepConv,
                                    RTDETRDecoder, Segment)
# Import tiện ích
try:
    from ultralytics.utils import LOGGER, colorstr
    from ultralytics.utils.ops import make_divisible
except ImportError:
    from ultralytics.utils import LOGGER, colorstr
    from ultralytics.nn.modules.utils import make_divisible

# ==========================================
# 1. ĐỊNH NGHĨA CLASS CSP TRỰC TIẾP
# ==========================================
class CSP(nn.Module):
    """CSP Bottleneck with 3 convolutions."""
    def __init__(self, c1, c2, n=1, shortcut=True, g=1, e=0.5):  # ch_in, ch_out, number, shortcut, groups, expansion
        super().__init__()
        c_ = int(c2 * e)  # hidden channels
        self.cv1 = Conv(c1, c_, 1, 1)
        self.cv2 = Conv(c1, c_, 1, 1)
        self.cv3 = Conv(2 * c_, c2, 1)  # optional act=FReLU(c2)
        self.m = nn.Sequential(*(Bottleneck(c_, c_, shortcut, g, e=1.0) for _ in range(n)))

    def forward(self, x):
        return self.cv3(torch.cat((self.m(self.cv1(x)), self.cv2(x)), 1))

# "Tiêm" CSP vào module tasks để hệ thống nhận diện được
tasks.CSP = CSP
print("✅ Đã định nghĩa và inject class CSP thủ công.")

# ==========================================
# 2. GHI ĐÈ HÀM PARSE_MODEL
# ==========================================
def custom_parse_model(d, ch, verbose=True):  # model_dict, input_channels(3)
    # Args
    max_channels = float('inf')
    nc, act, scales = (d.get(x) for x in ('nc', 'activation', 'scales'))
    depth, width, kpt_shape = (d.get(x, 1.0) for x in ('depth_multiple', 'width_multiple', 'kpt_shape'))
    if scales:
        scale = d.get('scale')
        if not scale:
            scale = tuple(scales.keys())[0]
            LOGGER.warning(f"WARNING ⚠️ no model scale passed. Assuming scale='{scale}'.")
        depth, width, max_channels = scales[scale]

    if act:
        Conv.default_act = eval(act)
        if verbose:
            LOGGER.info(f"{colorstr('activation:')} {act}")

    if verbose:
        LOGGER.info(f"\n{'':>3}{'from':>20}{'n':>3}{'params':>10}  {'module':<45}{'arguments':<30}")
    ch = [ch]
    layers, save, c2 = [], [], ch[-1]  # layers, savelist, ch out
    
    # === DANH SÁCH MODULE HỖ TRỢ (Đã thêm CSP) ===
    # Lưu ý: Thêm tasks.CSP vào danh sách này
    SUPPORTED_MODULES = (Classify, Conv, ConvTranspose, GhostConv, Bottleneck, GhostBottleneck, SPP, SPPF, DWConv, Focus,
                 BottleneckCSP, C1, C2, C2f, C3, C3TR, C3Ghost, nn.ConvTranspose2d, DWConvTranspose2d, C3x, RepC3, 
                 tasks.CSP) 
    
    # === DANH SÁCH MODULE CÓ THAM SỐ REPEAT (n) ===
    REPEAT_MODULES = (BottleneckCSP, C1, C2, C2f, C3, C3TR, C3Ghost, C3x, RepC3, 
                      tasks.CSP)

    for i, (f, n, m, args) in enumerate(d['backbone'] + d['head']):
        # Logic tìm module: Ưu tiên tìm trong tasks (nơi đã inject CSP)
        m_str = m
        m = getattr(torch.nn, m[3:]) if 'nn.' in m else getattr(tasks, m)
        
        for j, a in enumerate(args):
            if isinstance(a, str):
                with contextlib.suppress(ValueError):
                    args[j] = locals()[a] if a in locals() else ast.literal_eval(a)

        n = n_ = max(round(n * depth), 1) if n > 1 else n
        
        if m in SUPPORTED_MODULES:
            c1, c2 = ch[f], args[0]
            if c2 != nc:
                c2 = make_divisible(min(c2, max_channels) * width, 8)

            args = [c1, c2, *args[1:]]
            if m in REPEAT_MODULES:
                args.insert(2, n)
                n = 1
        elif m is AIFI:
            args = [ch[f], *args]
        elif m in (HGStem, HGBlock):
            c1, cm, c2 = ch[f], args[0], args[1]
            args = [c1, cm, c2, *args[2:]]
            if m is HGBlock:
                args.insert(4, n)
                n = 1
        elif m is nn.BatchNorm2d:
            args = [ch[f]]
        elif m is Concat:
            c2 = sum(ch[x] for x in f)
        elif m in (Detect, Segment, Pose, RTDETRDecoder):
            args.append([ch[x] for x in f])
            if m is Segment:
                args[2] = make_divisible(min(args[2], max_channels) * width, 8)
        else:
            c2 = ch[f]

        m_ = nn.Sequential(*(m(*args) for _ in range(n))) if n > 1 else m(*args)
        t = str(m)[8:-2].replace('__main__.', '')
        m.np = sum(x.numel() for x in m_.parameters())
        m_.i, m_.f, m_.type = i, f, t
        if verbose:
            LOGGER.info(f'{i:>3}{str(f):>20}{n_:>3}{m.np:10.0f}  {t:<45}{str(args):<30}')
        save.extend(x % i for x in ([f] if isinstance(f, int) else f) if x != -1)
        layers.append(m_)
        if i == 0:
            ch = []
        ch.append(c2)
    return nn.Sequential(*layers), sorted(save)

# 3. ÁP DỤNG HÀM MỚI
tasks.parse_model = custom_parse_model
print("✅ Đã cập nhật parse_model thành công.")

✅ Đã định nghĩa và inject class CSP thủ công.
✅ Đã cập nhật parse_model thành công.


In [18]:
# Initialize model from custom YAML (builds model from scratch with custom layers)
# Note: We initialize from YAML, not from yolov8n.pt, because the architecture is different
print("Initializing GE-YOLOv8 model from custom architecture...")
model = YOLO(modified_model_yaml)

# Print model information
print("\nModel Information:")
model.info()

WARNING ⚠️ no model scale passed. Assuming scale='l'.

                   from  n    params  module                                       arguments                     
  0                  -1  1      1856  ultralytics.nn.modules.conv.Conv             [3, 64, 3, 2]                 
  1                  -1  1     73984  ultralytics.nn.modules.conv.Conv             [64, 128, 3, 2]               


Initializing GE-YOLOv8 model from custom architecture...


  2                  -1  3    255232  CSP                                          [128, 128, 3, True]           
  3                  -1  1    295424  ultralytics.nn.modules.conv.Conv             [128, 256, 3, 2]              
  4                  -1  6   1904640  CSP                                          [256, 256, 6, True]           
  5                  -1  1   1180672  ultralytics.nn.modules.conv.Conv             [256, 512, 3, 2]              
  6                  -1  6   7610368  CSP                                          [512, 512, 6, True]           
  7                  -1  1   2360320  ultralytics.nn.modules.conv.Conv             [512, 512, 3, 2]              
  8                  -1  3   4068352  CSP                                          [512, 512, 3, True]           
  9                  -1  1    656896  ultralytics.nn.modules.block.SPPF            [512, 512, 5]                 
 10                  -1  1         0  torch.nn.modules.upsampling.Upsample         [None


Model Information:


(389, 40953369, 40953353, 154.71621120000003)

In [19]:
import torch
import torch.serialization

# --- BƯỚC 1: SỬA LỖI RECURSION ERROR ---
# Khôi phục hàm load gốc trực tiếp từ module serialization (nơi chứa code gốc)
# Điều này giúp cắt đứt vòng lặp vô tận do chạy cell patch nhiều lần
if hasattr(torch.serialization, 'load'):
    torch.load = torch.serialization.load
    print("🔄 Đã khôi phục torch.load gốc thành công.")

# --- BƯỚC 2: APPLY PATCH AN TOÀN ---
# Định nghĩa hàm patch gọi thẳng vào torch.serialization.load
def patched_torch_load(*args, **kwargs):
    # Ép buộc tắt chế độ weights_only để load được model custom
    kwargs['weights_only'] = False
    return torch.serialization.load(*args, **kwargs)

# Ghi đè
torch.load = patched_torch_load
print("✅ Đã patch torch.load (Chế độ an toàn).")

🔄 Đã khôi phục torch.load gốc thành công.
✅ Đã patch torch.load (Chế độ an toàn).


In [20]:
import glob
import os

# Đường dẫn tới thư mục labels
label_dir = '/kaggle/working/datasets/axial_t2/train/labels'
label_files = glob.glob(os.path.join(label_dir, '*.txt'))

print(f"Tìm thấy {len(label_files)} file .txt trong {label_dir}")

if len(label_files) > 0:
    print("Nội dung 5 file đầu tiên:")
    for f in label_files[:10]:
        with open(f, 'r') as file:
            content = file.read()
            print(f"--- {os.path.basename(f)} ---")
            print(content if content.strip() else "[FILE RỖNG]")
else:
    print("❌ Không tìm thấy file .txt nào! Hãy kiểm tra lại quy trình chuẩn bị dữ liệu.")

Tìm thấy 27962 file .txt trong /kaggle/working/datasets/axial_t2/train/labels
Nội dung 5 file đầu tiên:
--- 3488486405_2659601946_5.txt ---
1 0.567787 0.548214 0.083333 0.083333
2 0.443690 0.552764 0.083333 0.083333
--- 2975955864_2471804007_16.txt ---
1 0.562244 0.508892 0.083333 0.083333
2 0.455972 0.511905 0.083333 0.083333
--- 335455502_790180412_35.txt ---
1 0.572937 0.543956 0.083333 0.083333
--- 1646440651_536239867_24.txt ---
1 0.507965 0.471239 0.083333 0.083333
2 0.432416 0.467258 0.083333 0.083333
--- 1456430985_333208349_18.txt ---
1 0.552698 0.413735 0.083333 0.083333
2 0.473527 0.407477 0.083333 0.083333
--- 1812153363_2562001092_11.txt ---
1 0.539823 0.518142 0.083333 0.083333
2 0.464275 0.512444 0.083333 0.083333
--- 3801228235_929952248_30.txt ---
1 0.528620 0.517493 0.083333 0.083333
2 0.428319 0.518142 0.083333 0.083333
--- 1332176994_1030181512_11.txt ---
1 0.535948 0.480737 0.083333 0.083333
--- 712073652_239356302_19.txt ---
1 0.508413 0.505024 0.083333 0.083333
2

## 4. Training (Benchmark Settings)

Training with the same settings as the YOLOv11 experiment for fair comparison:
- `imgsz=384`
- `epochs=50`
- `batch=16`
- `optimizer='AdamW'`
- `lr0=0.001`

In [21]:
# Training configuration - same as YOLOv11 benchmark
TRAIN_CONFIG = {
    'data': data_yaml_path,
    'imgsz': 384,
    'epochs': 50,
    'batch': 16,
    'optimizer': 'AdamW',
    'lr0': 0.001,
    'project': 'ge_yolov8_benchmark',
    'name': 'axial_t2_run',
    'exist_ok': True,
    'pretrained': False,  # Train from scratch with custom architecture
    'verbose': True,
    'device': 0,  # Use GPU,
    'save':True,                    # Save checkpoints
    'save_period': 10,          # Save every 10 epochs
    'exist_ok': True
}

print("Training Configuration:")
for key, value in TRAIN_CONFIG.items():
    print(f"  {key}: {value}")

Training Configuration:
  data: /kaggle/working/datasets/axial_t2/data.yaml
  imgsz: 384
  epochs: 50
  batch: 16
  optimizer: AdamW
  lr0: 0.001
  project: ge_yolov8_benchmark
  name: axial_t2_run
  exist_ok: True
  pretrained: False
  verbose: True
  device: 0
  save: True
  save_period: 10


In [22]:
# [OPTIONAL] TensorBoard Visualization
# Uncomment the lines below if you want to view training metrics in TensorBoard
# Note: This cell is optional and can be skipped without affecting training

# %load_ext tensorboard
# %tensorboard --logdir ge_yolov8_benchmark

In [23]:
# Start training
print("\n" + "="*60)
print("Starting GE-YOLOv8 Training on RSNA 2024 Axial T2 Dataset")
print("="*60 + "\n")

results = model.train(**TRAIN_CONFIG)

New https://pypi.org/project/ultralytics/8.4.12 available 😃 Update with 'pip install -U ultralytics'



Starting GE-YOLOv8 Training on RSNA 2024 Axial T2 Dataset



Ultralytics YOLOv8.0.200 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla P100-PCIE-16GB, 16269MiB)
engine/trainer: task=detect, mode=train, model=/kaggle/working/datasets/axial_t2/yolov8-CSP-3classn.yaml, data=/kaggle/working/datasets/axial_t2/data.yaml, epochs=50, patience=50, batch=16, imgsz=384, save=True, save_period=10, cache=False, device=0, workers=8, project=ge_yolov8_benchmark, name=axial_t2_run, exist_ok=True, pretrained=False, optimizer=AdamW, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, show=False, save_txt=False, save_conf=False, save_crop=False, show_labels=True, show_conf=True, vid_stride=1, stream_buffer=False, line_width=None, visualize=False, augment=False, agnosti

KeyboardInterrupt: 

## 5. Comparison Output

Display validation metrics for comparison with YOLOv11 results.

In [ ]:
# Load the best model for validation
best_model_path = f"{TRAIN_CONFIG['project']}/{TRAIN_CONFIG['name']}/weights/best.pt"
best_model = YOLO(best_model_path)

# Run validation
print("\nRunning final validation on best model...")
val_results = best_model.val(data=data_yaml_path, imgsz=384, batch=16)

In [ ]:
# Print comparison metrics
print("\n" + "="*70)
print("GE-YOLOv8 BENCHMARK RESULTS - RSNA 2024 Axial T2")
print("="*70)
print(f"\nModel Architecture: yolov8-ECA-CSP (CSP + ECAAttention)")
print(f"Dataset: RSNA 2024 Lumbar Spine - Axial T2 Only")
print(f"Image Size: {TRAIN_CONFIG['imgsz']}")
print(f"Epochs: {TRAIN_CONFIG['epochs']}")
print(f"Batch Size: {TRAIN_CONFIG['batch']}")
print(f"Optimizer: {TRAIN_CONFIG['optimizer']}")
print(f"Learning Rate: {TRAIN_CONFIG['lr0']}")

print("\n" + "-"*70)
print("VALIDATION METRICS (for comparison with YOLOv11):")
print("-"*70)

# Extract key metrics
metrics = val_results.results_dict

print(f"\n{'Metric':<30} {'Value':<15}")
print("-" * 45)
print(f"{'mAP50':<30} {metrics.get('metrics/mAP50(B)', 'N/A'):<15.4f}")
print(f"{'mAP50-95':<30} {metrics.get('metrics/mAP50-95(B)', 'N/A'):<15.4f}")
print(f"{'Precision':<30} {metrics.get('metrics/precision(B)', 'N/A'):<15.4f}")
print(f"{'Recall':<30} {metrics.get('metrics/recall(B)', 'N/A'):<15.4f}")

print("\n" + "-"*70)
print("PER-CLASS METRICS:")
print("-"*70)

# Per-class AP if available
for i, class_name in enumerate(cfg.CLASS_NAMES):
    ap50_key = f'metrics/mAP50(B)'
    print(f"  Class {i} ({class_name}): See detailed results above")

print("\n" + "="*70)
print("TRAINING COMPLETE!")
print("="*70)
print(f"\nBest model saved at: {best_model_path}")
print(f"Results directory: {TRAIN_CONFIG['project']}/{TRAIN_CONFIG['name']}")

In [ ]:
# Display training curves
from IPython.display import Image, display
import os

results_dir = f"{TRAIN_CONFIG['project']}/{TRAIN_CONFIG['name']}"

# Display results.png if it exists
results_img = f"{results_dir}/results.png"
if os.path.exists(results_img):
    print("Training Curves:")
    display(Image(filename=results_img))

# Display confusion matrix if it exists
confusion_matrix_img = f"{results_dir}/confusion_matrix.png"
if os.path.exists(confusion_matrix_img):
    print("\nConfusion Matrix:")
    display(Image(filename=confusion_matrix_img))

## Summary

This notebook trained the **GE-YOLOv8** model (with CSP and ECAAttention modules) on the RSNA 2024 Axial T2 dataset.

**Key Results:**
- The validation metrics (mAP50, mAP50-95) can now be directly compared with YOLOv11 results
- The data pipeline and training settings were kept identical for fair comparison

**Files Generated:**
- Dataset: `/kaggle/working/datasets/axial_t2/`
- Model weights: `/kaggle/working/ge_yolov8_benchmark/axial_t2_run/weights/best.pt`
- Training logs: `/kaggle/working/ge_yolov8_benchmark/axial_t2_run/`